# Crowdcal Phase 4: fine-tune `mbert-ce` (ModernBERT-large + 2-class head, cross-entropy)

PREREG §4/§5. Kaggle: GPU T4 x2, Internet on. Trains on binary hard labels from `train@N`, picks the epoch with the
lowest raw soft Brier on `dev` (5-annotator entailment share), and saves one checkpoint per run to
`/kaggle/working/checkpoints/mbert-ce@<size>-s<seed>/` with `train_meta.json`. See `notebooks/README.md` for run order and splitting `RUNS` across sessions.

In [ ]:
# Only the train@* splits and dev are ever loaded in this notebook (PREREG §5: selection on dev only).
# The other two splits (recalibration + final evaluation) are never loaded here; `load()` below asserts it.
import gc, json, math, os, random, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path

DRY_RUN = False   # True = local smoke test only (48 train items, 50 dev items, 2 epochs)
CROWDCAL_COMMIT = "38cd6d871c5063bb2cf4f50670b2e66d6e4c76ea"
BASE, REV = os.environ.get("CROWDCAL_BASE", "answerdotai/ModernBERT-large"), "45bb4654a4d5aaff24dd11d4781fa46d39bf8c13"
OUT = Path(os.environ.get("CROWDCAL_OUT", "/kaggle/working/checkpoints"))

# (train size, seed). Set SELECT to a subset to split the work across sessions, e.g. SELECT = [(100, 0), (100, 1), (100, 2)].
RUNS = [(100, 0), (100, 1), (100, 2), (1000, 0), (1000, 1), (1000, 2), (10000, 0), (30000, 0)]
SELECT = None

# PREREG §5. Unspecified there, so fixed here and recorded in train_meta.json: weight_decay (torch AdamW default), grad clip.
HP = dict(lr=2e-5, batch_size=32, min_epochs=3, min_steps=300, warmup_frac=0.1, max_length=256,
          weight_decay=0.01, max_grad_norm=1.0, optimizer="AdamW", schedule="linear")
SPLIT = {100: "train@100", 1000: "train@1k", 10000: "train@10k", 30000: "train@30k"}
ALLOWED = set(SPLIT.values()) | {"dev"}

In [ ]:
# Setup: clone crowdcal at the pinned commit and install it (no deps: Kaggle already has torch/transformers).
if Path("/kaggle").exists():
    if not Path("crowdcal").exists() and not Path("pyproject.toml").exists():
        subprocess.run(["git", "clone", "https://github.com/AmanSinghNp/crowdcal"], check=True)
        subprocess.run(["git", "checkout", CROWDCAL_COMMIT], cwd="crowdcal", check=True)
    if Path("crowdcal/pyproject.toml").exists():
        os.chdir("crowdcal")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
    sys.path.insert(0, os.getcwd())
HEAD = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert DRY_RUN or HEAD == CROWDCAL_COMMIT, f"crowdcal at {HEAD}, expected {CROWDCAL_COMMIT}"

import numpy as np, torch, torch.nn.functional as F, transformers
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from crowdcal.data import load_split
assert tuple(int(x) for x in transformers.__version__.split(".")[:2]) >= (4, 48), "ModernBERT needs transformers>=4.48: pip install -U transformers, then restart"

def load(name):
    assert name in ALLOWED, f"{name!r}: only train@* and dev may be loaded in this notebook"
    return load_split(name)

dev_type = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device = torch.device(dev_type)
N_GPU = max(1, torch.cuda.device_count()) if dev_type == "cuda" else 1
FP16 = dev_type == "cuda"   # T4 has no bf16: fp16 autocast + GradScaler
print(dev_type, "GPUs:", N_GPU, "| torch", torch.__version__, "| transformers", transformers.__version__, "| crowdcal", HEAD[:8])

DEV = load("dev")
if DRY_RUN:
    DEV = DEV[:50]
DEV_Q = torch.tensor([i.q for i in DEV])
_rev = None if Path(BASE).is_dir() else REV
TOK = AutoTokenizer.from_pretrained(BASE, revision=_rev)

In [ ]:
def enc(items, idx):
    """(premise, hypothesis) pair, truncation only; same call as crowdcal.arms.LocalArm seqcls inference (plus max_length + padding)."""
    e = TOK([items[i].premise for i in idx], [items[i].hypothesis for i in idx], truncation=True,
            max_length=HP["max_length"], padding=True, return_tensors="pt")
    return {k: v.to(device) for k, v in e.items()}

@torch.no_grad()
def dev_soft_brier(net):
    """Mean (p - q)^2 over all dev items, p = softmax(logits)[1]. Batched, fp16 on cuda."""
    net.eval()
    order = sorted(range(len(DEV)), key=lambda i: len(DEV[i].premise) + len(DEV[i].hypothesis))
    p = torch.empty(len(DEV))
    for s in range(0, len(order), 128):
        idx = order[s:s + 128]
        with torch.autocast(dev_type, dtype=torch.float16, enabled=FP16):
            logits = net(**enc(DEV, idx)).logits
        p[idx] = torch.softmax(logits.float(), -1)[:, 1].cpu()
    return float(((p - DEV_Q) ** 2).mean())

def sync():
    if dev_type == "cuda":
        torch.cuda.synchronize()
    return time.perf_counter()

def now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def train_run(size, seed):
    name = f"mbert-ce@{size}-s{seed}"
    out = OUT / name
    if (out / "train_meta.json").exists():
        print("skip (finished):", name)
        return
    started = now()
    train = load(SPLIT[size])
    if DRY_RUN:
        train = train[:48]
    n, bs = len(train), HP["batch_size"]
    spe = math.ceil(n / bs)                                   # steps per epoch (last batch may be smaller)
    epochs = max(HP["min_epochs"], math.ceil(HP["min_steps"] / spe))   # "3 epochs or 300 steps, whichever is larger"
    if DRY_RUN:
        epochs = 2
    total = epochs * spe

    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)   # also seeds the head init
    cfg = AutoConfig.from_pretrained(BASE, revision=_rev, num_labels=2)
    if hasattr(cfg, "reference_compile"):   # transformers 4.x: skip torch.compile paths (unsafe under DataParallel)
        cfg.reference_compile = False
    model = AutoModelForSequenceClassification.from_pretrained(BASE, revision=_rev, config=cfg, attn_implementation="sdpa").to(device)
    net = torch.nn.DataParallel(model) if N_GPU > 1 else model   # batch 32 is split across GPUs: effective batch stays 32
    opt = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
    sched = get_linear_schedule_with_warmup(opt, math.ceil(HP["warmup_frac"] * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=FP16)
    gen = torch.Generator().manual_seed(seed)                 # data order

    hist, best, best_sd, best_ep, step, train_sec = [], math.inf, None, None, 0, 0.0
    for ep in range(1, epochs + 1):
        net.train()
        perm = torch.randperm(n, generator=gen).tolist()
        t0 = sync()
        for s in range(0, n, bs):
            idx = perm[s:s + bs]
            y = torch.tensor([train[i].label for i in idx], device=device)
            with torch.autocast(dev_type, dtype=torch.float16, enabled=FP16):
                logits = net(**enc(train, idx)).logits
            loss = F.cross_entropy(logits.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), HP["max_grad_norm"])
            scaler.step(opt); scaler.update(); sched.step()
            step += 1
        train_sec += sync() - t0
        b = dev_soft_brier(net)                               # not counted in gpu_hours
        hist.append({"epoch": ep, "step": step, "dev_soft_brier": b})
        if b < best:                                          # strict: ties keep the earliest epoch
            best, best_ep = b, ep
            best_sd = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
        print(f"{name} epoch {ep}/{epochs} step {step}/{total} loss {loss.item():.4f} dev_brier {b:.5f} (best {best:.5f} @ {best_ep})", flush=True)

    model.load_state_dict(best_sd)
    del best_sd
    out.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out); TOK.save_pretrained(out)

    # Inference speed: single-item, unbatched, fp32, exactly the LocalArm path, on one GPU, 200 dev items.
    model.eval()
    sample = DEV[:20 if DRY_RUN else 200]
    def one(it):
        e = TOK(it.premise, it.hypothesis, return_tensors="pt", truncation=True).to(device)
        with torch.no_grad():
            return torch.softmax(model(**e).logits, dim=-1)[0, 1].item()
    for it in sample[:5]:
        one(it)
    t0 = sync()
    for it in sample:
        one(it)
    infer = (sync() - t0) / len(sample) * 1000

    meta = {
        "arm": "mbert-ce", "size": size, "seed": seed,
        "base": f"{BASE}@{REV}", "crowdcal_commit": HEAD,
        "hyperparameters": {**HP, "epochs": epochs, "steps_per_epoch": spe, "total_steps": total,
                            "warmup_steps": math.ceil(HP["warmup_frac"] * total), "n_train": n, "n_dev": len(DEV),
                            "n_gpu": N_GPU, "fp16": FP16, "attn_implementation": "sdpa", "dry_run": DRY_RUN},
        "epochs": hist, "selected_epoch": best_ep, "selected_dev_soft_brier": best,
        "gpu_hours": train_sec / 3600 * N_GPU,
        "infer_sec_per_1k": infer, "infer_precision": "fp32, single item, 1 GPU" if dev_type == "cuda" else f"fp32, single item, {dev_type}",
        "versions": {"python": sys.version.split()[0], "torch": torch.__version__, "transformers": transformers.__version__,
                     "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if dev_type == "cuda" else dev_type},
        "started": started, "finished": now(),
    }
    (out / "train_meta.json").write_text(json.dumps(meta, indent=2) + "\n")   # written last: its presence marks a finished run
    print(f"done {name}: epoch {best_ep}, dev brier {best:.5f}, {meta['gpu_hours']:.3f} GPU-h, {infer:.1f} s/1k")
    del model, net, opt, sched, scaler
    gc.collect()
    if dev_type == "cuda":
        torch.cuda.empty_cache()

def write_summary():
    rows = []
    for p in sorted(OUT.glob("*/train_meta.json")):
        m = json.loads(p.read_text())
        rows.append({"dir": p.parent.name, **{k: m[k] for k in ("arm", "size", "seed", "selected_epoch", "selected_dev_soft_brier",
                                                                "gpu_hours", "infer_sec_per_1k", "finished") if k in m}})
    (OUT / "summary.json").write_text(json.dumps(rows, indent=2) + "\n")
    return rows

In [ ]:
todo = [(100, 0)] if DRY_RUN else [r for r in RUNS if SELECT is None or r in SELECT]
OUT.mkdir(parents=True, exist_ok=True)
for size, seed in todo:
    train_run(size, seed)
    rows = write_summary()
for r in rows:
    print(r)

## Download the outputs

Each finished run is a directory `checkpoints/mbert-ce@<size>-s<seed>/` (weights, tokenizer, `train_meta.json`) plus `checkpoints/summary.json`.

1. Run the notebook with **Save Version -> Save & Run All (Commit)** (or keep the interactive session alive), then open the notebook's **Output** tab.
2. Download the run directories (Output tab, or `kaggle kernels output <user>/<notebook-slug> -p kaggle_out`).
3. Put each run directory at `data/checkpoints/<dir>/` in your local crowdcal checkout, e.g. `data/checkpoints/mbert-ce@100-s0/` (must contain `config.json`, `model.safetensors`, tokenizer files, `train_meta.json`). `data/checkpoints/` is git-ignored.
4. Check: `ls data/checkpoints/*/train_meta.json | wc -l` equals the number of runs you trained.

If you ran `RUNS` in several sessions, repeat for each session; directories have unique names, so nothing collides.